In [ ]:
# torch, faiss (k-NN/perplexity graph construction) and PHATE's numba-based internals
# all get loaded into one process here; with unconstrained thread pools that combination
# makes macOS abort with a duplicate-OpenMP-runtime error (kernel crash / segfault).
# Capping thread counts before any of those libraries import, plus the (unsafe but
# standard) escape hatch as a second line of defense, avoids it.
import os
os.environ.setdefault('KMP_DUPLICATE_LIB_OK', 'TRUE')
os.environ.setdefault('OMP_NUM_THREADS', '1')
os.environ.setdefault('MKL_NUM_THREADS', '1')
os.environ.setdefault('OPENBLAS_NUM_THREADS', '1')
os.environ.setdefault('NUMEXPR_NUM_THREADS', '1')
os.environ.setdefault('NUMBA_NUM_THREADS', '1')
import sys
import json
import random
import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt

import napari
from scipy.interpolate import griddata
from pathlib import Path


project_root = Path.cwd().parent.parent
sys.path.insert(0, str(project_root / 'src'))
sys.path.insert(0, str(Path.cwd().parent / 'scripts'))

sys.path.append(f"{project_root}src/")
# Import utilities
from utilities import KernelNN, KernelNN_subgraph, GraphLMDBReader, setup_device, get_output_affinities
from plot_utilities import plot_results_napari, plot_results_matplotlib

sys.path.append(f"{project_root}/ERA5/scripts/")
from plot_utilities import *
from utilities_era5 import ERA5_Graph_Generator as Data_Generator
from utilities_era5 import ERA5_Graph_Generator_perSampleNorm as Data_Generator_old
data_path = project_root / 'ERA5' / 'data'

# Device setup
device = setup_device()
print(f'Using device: {device}')

from sklearn.decomposition import PCA
from sklearn.manifold import TSNE
import phate
import subprocess


# UMAP and Concord only exist in their own conda envs (the env running this notebook has
# phate but not umap/anndata/concord), so both are invoked as subprocesses: each reads
# <buffer_id>.npy from its workdir and writes res_<buffer_id>.npy there. The buffer id is
# per-process so two concurrent runs cannot race on the same filenames.
def external_embed(env, workdir, script, buffer_id, X):
    np.save(f'{workdir}/{buffer_id}.npy', X)
    subprocess.run(f'cd {workdir} && conda run -n {env} python {script} {buffer_id}', shell=True)
    Y = np.load(f'{workdir}/res_{buffer_id}.npy')
    os.remove(f'{workdir}/{buffer_id}.npy')
    os.remove(f'{workdir}/res_{buffer_id}.npy')
    return Y

from scipy.spatial import cKDTree

def load_model(id:str, model_type:str='random_crop'):
    model_path = project_root / 'ERA5' / 'trained_models' / model_type / f'{id}.pth'
    config_path = project_root / 'ERA5' / 'trained_models' / model_type / 'logs' / 'configs' / f'{id}.json' 
    
    with open(config_path, 'r') as f:
        config = json.load(f)
    
    model = KernelNN(
        width=config['width'],
        ker_width=config['ker_width'],
        depth=config['depth'],
        ker_in=config['edge_attr'],
        in_width=config['node_features'],
        out_width=config['out_width']
    ).to(device)
    
    model.load_state_dict(torch.load(model_path, map_location=device))
    model.eval()
    
    return model, config


# ERA5_Graph_Generator.generate_normed_sample() draws the sampled point subset with
# np.random.choice, and nothing in the sampling path re-seeds, so the sampled graph -- and
# every embedding, statistic and figure derived from it -- changes run-to-run unless the
# global RNGs are seeded first. The in-notebook methods pass random_state=42 explicitly and
# the UMAP/concord runner scripts hardcode their own seed, so seeding here is what makes
# the sampled input itself reproducible.
SEED = 42


def set_seed(seed: int = SEED):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)


set_seed()


----
### load model + dataset 

In [ ]:
model, config = load_model(id='regular_papermodel2_2026-05-04_t12-01-43' , model_type='regular')#test1_model5_median_tkernel_2026-04-22_t21-22-37

In [ ]:
lat_min = config["lat_min"]
lat_max = config["lat_max"]
lon_min = config["lon_min"]
lon_max = config["lon_max"]
#N = config["N"]
k = config["k"]
perplexity = config["perplexity"]
sampling_method = config["sampling_method"]
train_date_start_end = config["train_date_start_end"]
test_date_start_end = config["test_date_start_end"]
train_samples_per_date = config["train_samples_per_date"]
test_samples_per_date = config["test_samples_per_date"]
feature_mode = config["feature_mode"]
vars = config["vars"]


In [ ]:
lat_range = (lat_max - lat_min)/2
lon_range = (lon_max - lon_min)/2
margin=5
side_crop=7
'''
bbox1 = lon_min, lat_min+lat_range-margin, lon_max-lon_range+margin, lat_max
bbox2 = lon_min+lon_range-margin, lat_min+lat_range-margin, lon_max, lat_max
bbox3 = lon_min, lat_min, lon_max-lon_range+margin, lat_max-lat_range+margin
bbox4 = lon_min+lon_range-margin, lat_min, lon_max, lat_max-lat_range+margin

bboxes = [bbox1, bbox2, bbox3, bbox4]
'''
bbox1 = lon_min+side_crop, lat_min+side_crop, lon_max-lon_range+margin, lat_max-side_crop
bbox2 = lon_min+lon_range-margin, lat_min+side_crop, lon_max-side_crop, lat_max-side_crop
bboxes = [bbox1, bbox2]

In [ ]:
def get_landmarks(locations_1, locations_2):
    """ 
    Find common locations between two datasets based on lat and lon column.
    Returns list of touples containing the respective indices of the shared points in the two datasets
    """
    matches = (
        locations_1.reset_index(names="idx1")
        .merge(
            locations_2.reset_index(names="idx2"),
            on=["lat", "lon"]
        )
    )
    matched_indices = np.array(matches[['idx1', 'idx2']].to_numpy())
    return matched_indices

In [ ]:
def align(reference, moving, landmarks, scaling=True, allow_reflection=False):
    """
    Align 'moving' to 'reference' using Procrustes analysis.

    Parameters
    ----------
    reference : (N, d) ndarray
        Reference embedding.

    moving : (M, d) ndarray
        Embedding to be transformed.

    landmarks : list of (i_ref, i_mov)
        Corresponding landmark indices.
        i_ref indexes rows of 'reference'.
        i_mov indexes rows of 'moving'.

    scaling : bool, default=True
        Whether to estimate a uniform scaling factor.

    allow_reflection : bool, default=False
        If False, reflections are prevented.

    Returns
    -------
    aligned : (M, d) ndarray
        The transformed moving embedding.

    R : (d, d) ndarray
        Rotation matrix.

    s : float
        Scale factor.

    t : (d,) ndarray
        Translation vector.
    """

    if len(landmarks) < 2:
        raise ValueError("Need at least two landmark correspondences.")

    reference = np.asarray(reference, dtype=float)
    moving = np.asarray(moving, dtype=float)

    ref_idx = np.array([i for i, _ in landmarks])
    mov_idx = np.array([j for _, j in landmarks])

    X = reference[ref_idx]
    Y = moving[mov_idx]

    if X.shape[1] != Y.shape[1]:
        raise ValueError("Embeddings must have the same dimension.")

    # Compute centroids
    muX = X.mean(axis=0)
    muY = Y.mean(axis=0)

    # Center the landmark sets
    Xc = X - muX
    Yc = Y - muY

    # Cross-covariance
    H = Yc.T @ Xc

    # SVD
    U, S, Vt = np.linalg.svd(H)

    # Optimal rotation
    R = U @ Vt

    # Prevent reflections if requested
    if not allow_reflection and np.linalg.det(R) < 0:
        U[:, -1] *= -1
        R = U @ Vt

    # Uniform scale
    if scaling:
        s = S.sum() / np.sum(Yc ** 2)
    else:
        s = 1.0

    # Translation
    t = muX - s * (muY @ R)

    # Transform entire moving embedding
    aligned = s * (moving @ R) + t

    return aligned, R, s, t

In [ ]:
nofe_mean_dist_per_pred = []
pca_mean_dist_per_pred = []
tsne_mean_dist_per_pred = []
umap_mean_dist_per_pred = []
phate_mean_dist_per_pred = []
spatialPCA_mean_dist_per_pred = []
concord_mean_dist_per_pred = []

# Start the spatialPCA R process ONCE, outside the loop -- library(SpatialPCA) (Seurat,
# SPARK, etc.) is the slow part to load, and conda run's own activation overhead also
# used to get paid on every single call. It now stays alive and answers one request per
# line over stdin/stdout for the whole loop below.
spdir = f'{project_root}/ERA5/spatialpca'
# Unique per process so concurrently running scripts/notebooks don't collide on the
# same buffer filenames (a fixed call_id caused FileNotFoundError / silent data
# corruption when two spatialPCA calls raced on the same X.csv/coords.csv paths).
spatialpca_call_id = f'spatialpca_buffer_{os.getpid()}'
spatialpca_proc = subprocess.Popen(
    ['conda', 'run', '-n', 'spatialpca', '--no-capture-output', 'Rscript', 'execute_spatialpca_server.R'],
    cwd=spdir, stdin=subprocess.PIPE, stdout=subprocess.PIPE, stderr=subprocess.PIPE,
    text=True, bufsize=1,
    # R searches the *user* library (~/Library/R/<arch>/<ver>/library) before the conda
    # env's own library. CRAN-built packages there link the CRAN R framework rather than
    # conda's libR, and SpatialPCA imports Rcpp -- whose foreign .so then segfaults at
    # dyn.load. Pointing R_LIBS_USER at a nonexistent path lets the env's own packages win.
    env=dict(os.environ, R_LIBS_USER='/nonexistent'),
)
ready = spatialpca_proc.stdout.readline().strip()
if ready != 'READY':
    raise RuntimeError(f'spatialPCA server failed to start: {ready!r}\n{spatialpca_proc.stderr.read()}')
print('spatialPCA server ready.')


def spatialpca_embed(X, coords, call_id=spatialpca_call_id, n_components=3):
    np.savetxt(f'{spdir}/{call_id}_X.csv', X, delimiter=',')
    np.savetxt(f'{spdir}/{call_id}_coords.csv', coords, delimiter=',')

    spatialpca_proc.stdin.write(f'{call_id} {n_components}\n')
    spatialpca_proc.stdin.flush()
    response = spatialpca_proc.stdout.readline().strip()

    os.remove(f'{spdir}/{call_id}_X.csv')
    os.remove(f'{spdir}/{call_id}_coords.csv')

    if response != 'OK':
        raise RuntimeError(f'spatialPCA failed for {call_id}: {response}')

    Y = np.loadtxt(f'{spdir}/res_{call_id}.csv', delimiter=',', skiprows=1)
    os.remove(f'{spdir}/res_{call_id}.csv')
    return Y

dates = pd.date_range("2019-01-01", "2019-01-31").strftime("%Y-%m-%d").tolist()

# Seed before the loop: generate_normed_sample() draws each patch subset with global
# np.random, so without this the sampled points -- and every alignment error below --
# differ on every run. Seeding once here reproduces the whole 31-date x patch sequence.
set_seed()

for date in dates:
    print(date)
        
    #Generator = Data_Generator_old(data_path)
    Generator = Data_Generator(data_path)

    N=10000

    graphs=[]
    locations_list=[]
    X_list=[]

    for bbox in bboxes:
        Generator.load_data(bbox=bbox, vars=vars, filename=f'{date}.nc')
        graph = Generator.sample_graph(N=N, k=k, perplexity=perplexity, sampling_method=sampling_method, gen_feature_mode=feature_mode)
        sample = graph.to(device)
        graphs.append(sample)
        locations_list.append(pd.DataFrame({"lat": sample.meta.gps[:,0], "lon": sample.meta.gps[:,1]}))
        X_list.append(sample.normed_values.detach().cpu().numpy())

    locations = pd.DataFrame(np.vstack(locations_list), columns=['lat', 'lon'])
    landmarks = get_landmarks(locations_list[0], locations_list[1])

    df1 = locations_list[0]
    df2 = locations_list[1]

    # Preserve indices as columns
    tmp1 = df1.reset_index().rename(columns={'index': 'idx1'})
    tmp2 = df2.reset_index().rename(columns={'index': 'idx2'})

    # Merge on identical lat/lon pairs
    matches = tmp1.merge(tmp2, on=['lat', 'lon'])

    i1 = matches['idx1'].to_numpy()
    i2 = matches['idx2'].to_numpy()

    pred_nofe_list = []
    pred_pca_list = []
    pred_tsne_list = []
    pred_umap_list = []
    pred_phate_list = []
    pred_spatialPCA_list = []
    pred_concord_list = []

    for sample in graphs:
        # NOFE
        print('NOFE ...', end='\r')
        pred_nofe = model(sample).detach().cpu().numpy()
        pred_nofe = (pred_nofe - pred_nofe.mean(axis=0)) / pred_nofe.std(axis=0)
        pred_nofe_list.append(pred_nofe)
        
        X = sample.normed_values.detach().cpu().numpy()
        
        # PCA
        print('PCA ...', end='\r')
        pca = PCA(n_components=3)
        pred_pca = pca.fit_transform(X)
        pred_pca = (pred_pca - pred_pca.mean(axis=0)) / pred_pca.std(axis=0)
        pred_pca_list.append(pred_pca)

        # t-SNE
        print('t-SNE ...', end='\r')
        pred_tsne = TSNE(
            n_components=3,
            perplexity=30,
            learning_rate="auto",
            random_state=42
        ).fit_transform(X)
        pred_tsne = (pred_tsne - pred_tsne.mean(axis=0)) / pred_tsne.std(axis=0)
        pred_tsne_list.append(pred_tsne)

        # UMAP
        print('UMAP ...', end='\r')
        pred_umap = external_embed(
            'umap', f'{project_root}/ERA5/umap', 'execute_umap.py',
            f'umap_buffer_{os.getpid()}', X,
        )
        pred_umap = (pred_umap - pred_umap.mean(axis=0)) / pred_umap.std(axis=0)
        pred_umap_list.append(pred_umap)

        # PHATE
        print('PHATE ...', end='\r')
        pred_phate = phate.PHATE(
            n_components=3, knn=15, mds_solver='smacof', random_state=42, verbose=False
        ).fit_transform(X)
        pred_phate = (pred_phate - pred_phate.mean(axis=0)) / pred_phate.std(axis=0)
        pred_phate_list.append(pred_phate)

        # spatialPCA
        print('spatialPCA ...', end='\r')
        pred_spatialPCA = spatialpca_embed(X, sample.normed_coords.detach().cpu().numpy())
        pred_spatialPCA = (pred_spatialPCA - pred_spatialPCA.mean(axis=0)) / pred_spatialPCA.std(axis=0)
        pred_spatialPCA_list.append(pred_spatialPCA)

        # Concord
        print('Concord ...', end='\r')
        pred_concord = external_embed(
            'concord', f'{project_root}/ERA5/concord', 'execute_concord.py',
            f'concord_buffer_{os.getpid()}', X,
        )
        pred_concord = (pred_concord - pred_concord.mean(axis=0)) / pred_concord.std(axis=0)
        pred_concord_list.append(pred_concord)

    aligned, _, _, _ = align(pred_pca_list[0], pred_pca_list[1], landmarks)
    pred_pca_list_aligned = [pred_pca_list[0], aligned]

    aligned, _, _, _ = align(pred_tsne_list[0], pred_tsne_list[1], landmarks)
    pred_tsne_list_aligned = [pred_tsne_list[0], aligned]
    
    aligned, _, _, _ = align(pred_umap_list[0], pred_umap_list[1], landmarks)
    pred_umap_list_aligned = [pred_umap_list[0], aligned]

    aligned, _, _, _ = align(pred_phate_list[0], pred_phate_list[1], landmarks)
    pred_phate_list_aligned = [pred_phate_list[0], aligned]

    aligned, _, _, _ = align(pred_spatialPCA_list[0], pred_spatialPCA_list[1], landmarks)
    pred_spatialPCA_list_aligned = [pred_spatialPCA_list[0], aligned]

    aligned, _, _, _ = align(pred_concord_list[0], pred_concord_list[1], landmarks)
    pred_concord_list_aligned = [pred_concord_list[0], aligned]

    nofe_mean_dist_per_pred.append(np.mean((pred_nofe_list[0][i1] - pred_nofe_list[1][i2])**2, axis=1).mean())
    pca_mean_dist_per_pred.append(np.mean((pred_pca_list_aligned[0][i1] - pred_pca_list_aligned[1][i2])**2, axis=1).mean())
    tsne_mean_dist_per_pred.append(np.mean((pred_tsne_list_aligned[0][i1] - pred_tsne_list_aligned[1][i2])**2, axis=1).mean())
    umap_mean_dist_per_pred.append(np.mean((pred_umap_list_aligned[0][i1] - pred_umap_list_aligned[1][i2])**2, axis=1).mean())
    phate_mean_dist_per_pred.append(np.mean((pred_phate_list_aligned[0][i1] - pred_phate_list_aligned[1][i2])**2, axis=1).mean())
    spatialPCA_mean_dist_per_pred.append(np.mean((pred_spatialPCA_list_aligned[0][i1] - pred_spatialPCA_list_aligned[1][i2])**2, axis=1).mean())
    concord_mean_dist_per_pred.append(np.mean((pred_concord_list_aligned[0][i1] - pred_concord_list_aligned[1][i2])**2, axis=1).mean())


# Shut the spatialPCA server down (normal path only; on error the run already fails loudly
# and the server is a child of this kernel).
spatialpca_proc.stdin.write('QUIT\n')
spatialpca_proc.stdin.flush()
spatialpca_proc.wait(timeout=10)

print(f'NOFE: {np.mean(nofe_mean_dist_per_pred):.3f} $\pm$ {np.std(nofe_mean_dist_per_pred):.3f}')
print(f'PCA:  {np.mean(pca_mean_dist_per_pred):.3f} $\pm$ {np.std(pca_mean_dist_per_pred):.3f}')
print(f't-SNE:{np.mean(tsne_mean_dist_per_pred):.3f} $\pm$ {np.std(tsne_mean_dist_per_pred):.3f}')
print(f'UMAP: {np.mean(umap_mean_dist_per_pred):.3f} $\pm$ {np.std(umap_mean_dist_per_pred):.3f}')
print(f'PHATE: {np.mean(phate_mean_dist_per_pred):.3f} $\pm$ {np.std(phate_mean_dist_per_pred):.3f}')
print(f'spatialPCA: {np.mean(spatialPCA_mean_dist_per_pred):.3f} $\pm$ {np.std(spatialPCA_mean_dist_per_pred):.3f}')
print(f'Concord: {np.mean(concord_mean_dist_per_pred):.3f} $\pm$ {np.std(concord_mean_dist_per_pred):.3f}')
